In [ ]:
from holoviews.operation.datashader import rasterize
from dask.distributed import Client
from mpire import WorkerPool
import dask.dataframe as ddf
from pathlib import Path
import dask.array as da
import multiprocessing
import holoviews as hv
import pandas as pd
import numpy as np

hv.extension('bokeh',inline=True)


In [ ]:
# Connect to Dask scheduler
# Option 1: Use environment variable (set by JupyterHub)
import os
scheduler_addr = os.environ.get('DASK_SCHEDULER_ADDRESS', 'simple-scheduler.dask.svc.cluster.local:8786')
client = Client(scheduler_addr)

# Option 2: Direct connection (uncomment if not using JupyterHub)
# client = Client("simple-scheduler.dask.svc.cluster.local:8786")

# Option 3: Use Dask threadpool scheduler (local, no cluster)
# client = Client()

In [ ]:
client

# Small sample problem using dask distributed arrays:  

In [ ]:
arr_small = da.random.random( (20_000,20_000), chunks = (1_000,1_000) )
arr_small

In [ ]:
image_small=hv.Image( (list(range(arr_small.shape[1])), list(range(arr_small.shape[0])), arr_small) )
rasterized_small = rasterize(image_small)

In [ ]:
hv.output(rasterized_small)

# Large sample problem using dask distributed arrays:  

In [ ]:
arr_large = da.random.random( (500_000,500_000), chunks = (20_000,10_000) )
arr_large

In [ ]:
image_large=hv.Image( (list(range(arr_large.shape[1])), list(range(arr_large.shape[0])), arr_large) )
rasterized_large = rasterize(image_large)

In [ ]:
hv.output(rasterized_large)

# Generate sample data stored in files: 
-  I'm more interested in HDF5 but using parquet files for now since that's what your demo used

In [ ]:
save_dir="/mnt/dhfo/temp_data"
N_steps_per_file=25000
N_channels=1_200
N_files=500
cols= list(np.arange(N_channels).astype(str))


In [ ]:
def make_file(ind):
    df = pd.DataFrame(np.random.normal(10,0.1,size=(N_steps_per_file, N_channels)),columns=cols)
    df.to_parquet(save_dir+'/file_'+str(ind)+'.parquet')#,compression='snappy')

with WorkerPool(n_jobs=multiprocessing.cpu_count()-1,pass_worker_id=False) as pool:
    _=pool.map(make_file,range(0, N_files),iterable_len=len(range(0, N_files)),progress_bar=True,concatenate_numpy_output=False)

file_list = sorted(list(Path(save_dir).glob('*.parquet')))    

In [ ]:
print("To Image..")
img_dask = hv.Image((np.arange(N_channels), np.arange(N_steps_per_file*len(file_list)), dask_df),kdims=['channels','time'])

x_size=1_200
y_size=500
rasterized_img_dask = rasterize(img_dask, width=x_size, height=y_size).opts(width=x_size, height=y_size, cmap='jet')



In [ ]:
hv.output(rasterized_img_dask)